In [ ]:
## detecting the local yaml files
# change from V1 to V2: saves the action yamls in a subfolder "actions".

In [2]:
import os
import re
import csv
from pathlib import Path

# === CONFIG ===
# IMPORTANT: point this to the *main* All_Config_Files that came from your extractor
ALL_CONFIG_DIR = Path(
    r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\All_Config_Files"
    # or whatever your main bucket path is
)

OUTPUT_CSV = Path(
    r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\Local_Instru_Tests\Repos_with_Local_GHA_Actions_NON_WORKFLOW.csv"
)
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)

# Regex to find "uses: ./something" (with or without leading '-' and with/without quotes)
LOCAL_USES_PATTERN = re.compile(
    r'^\s*(?:-\s*)?uses:\s*["\']?(\./[^\s"\']*)',
    re.MULTILINE,
)


def main():
    if not ALL_CONFIG_DIR.exists():
        print(f" ALL_CONFIG_DIR does not exist: {ALL_CONFIG_DIR}")
        return

    repos_with_local_actions = {}  # key: owner_repo, value: list[dict]

    total_files = 0
    total_yaml = 0
    total_gha_yaml = 0
    total_with_matches = 0

    print(f"🔁 Scanning config folder: {ALL_CONFIG_DIR}")

    for root, _, files in os.walk(ALL_CONFIG_DIR):
        for fname in files:
            total_files += 1
            lname = fname.lower()
            if not lname.endswith((".yml", ".yaml")):
                continue

            total_yaml += 1

            # Only GitHub Actions workflow configs (from extractor)
            # e.g. owner.repo__github_actions++something.yml
            if "__github_actions++" not in lname:
                continue

            total_gha_yaml += 1
            full_path = Path(root) / fname

            try:
                text = full_path.read_text(encoding="utf-8", errors="ignore")
            except Exception as e:
                print(f" Could not read {full_path}: {e}")
                continue

            raw_matches = LOCAL_USES_PATTERN.findall(text)

            # 🔍 NEW: keep only uses that are NOT pointing into .github/workflows
            # i.e. ignore things like "./.github/workflows/foo.yml"
            filtered_matches = []
            for m in raw_matches:
                norm = m.replace("\\", "/")  # normalize separators
                if "/.github/workflows/" in norm:
                    # This is a reusable workflow under .github/workflows → already extracted
                    continue
                filtered_matches.append(m)

            if not filtered_matches:
                continue  # no relevant local action reference in this workflow

            total_with_matches += 1

            # Everything before "__" is the "owner.repo" identifier
            owner_repo = fname.split("__", 1)[0]

            entry_list = repos_with_local_actions.setdefault(owner_repo, [])
            entry_list.append(
                {
                    "config_file_name": fname,                 # flattened workflow file
                    "full_path": str(full_path),
                    "uses_paths": ";".join(sorted(set(filtered_matches))),  # only NON-workflow targets
                }
            )

    # === DEBUG SUMMARY ===
    print("\n=== SCAN SUMMARY ===")
    print(f"Total files visited:                                 {total_files}")
    print(f"Total YAML files:                                    {total_yaml}")
    print(f"Total '__github_actions++' YAML files (workflows):   {total_gha_yaml}")
    print(f"Workflow files with NON-.github/workflows 'uses:':   {total_with_matches}")

    if not repos_with_local_actions:
        print("\n✅ No workflows with local GitHub Actions outside .github/workflows found.")
        return

    # 1) Console summary
    print("\n=== Repos with NON-.github/workflows local actions ===")
    print(f"Total repos: {len(repos_with_local_actions)}\n")

    for owner_repo, entries in sorted(repos_with_local_actions.items()):
        print(f"- {owner_repo} (workflow files: {len(entries)})")
        for e in entries:
            print(f"    • {e['config_file_name']}  →  {e['uses_paths']}")
        print()

    # 2) CSV for Step 2 (cloning & pulling action.yml, etc.)
    with OUTPUT_CSV.open("w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(
            [
                "owner_repo",
                "num_workflow_files",
                "config_file_name",
                "full_path",
                "uses_paths",   # only local targets NOT in .github/workflows
            ]
        )

        for owner_repo, entries in sorted(repos_with_local_actions.items()):
            num_files = len(entries)
            for idx, e in enumerate(entries):
                writer.writerow(
                    [
                        owner_repo,
                        num_files if idx == 0 else "",
                        e["config_file_name"],
                        e["full_path"],
                        e["uses_paths"],
                    ]
                )

    print(f"\n Details written to: {OUTPUT_CSV}")


if __name__ == "__main__":
    main()


🔁 Scanning config folder: C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ1\All_Config_Files

=== SCAN SUMMARY ===
Total files visited:                                 78894
Total YAML files:                                    12667
Total '__github_actions++' YAML files (workflows):   9724
Workflow files with NON-.github/workflows 'uses:':   499

=== Repos with NON-.github/workflows local actions ===
Total repos: 160

- 1q23lyc45.kitsunemagisk (workflow files: 1)
    • 1q23lyc45.kitsunemagisk__github_actions++android.yml  →  ./.github/actions/setup

- 4accccc.vivo-magisk-suu (workflow files: 1)
    • 4accccc.vivo-magisk-suu__github_actions++build.yml  →  ./.github/actions/setup

- alvr.katana (workflow files: 1)
    • alvr.katana__github_actions++katana.yml  →  ./.github/actions/common-steps

- anysoftkeyboard.anysoftkeyboard (workflow files: 4)
    • anysoftkeyboard.anysoftkeyboard__github_actions++checks.yml  →  ./.github/actions/collect-reports;./.github/actions/deploy-reques

In [ ]:
## Download the local yml files through a similar clone with the initial data extraction with the cut off date: Aug 10, 2025

In [1]:
# -*- coding: utf-8 -*-
from __future__ import annotations

import csv
import os
import re
import shutil
import stat
import subprocess
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, Optional, Set
from urllib.parse import urlparse

import pandas as pd
import requests
from dotenv import load_dotenv

# ========= CONFIG & PATHS =========

# Your env file location
ENV_DIR = Path(r"C:\GitHub\Android-Mobile-Apps")
ENV_FILE_NAME = "All_Tokens.env"  # keep this exact filename

# Cutoff date (ISO 8601, UTC)
CUTOFF_ISO = "2025-08-10T23:59:59Z"

# Base directory (drive root)
BASE_RQ1 = Path(r"D:\temp")  # <--- key: we treat D:\ as base

# URL list for all repos
URL_LIST_CSV = BASE_RQ1 / "URL_List.csv"

# Local instrumentation detection results (where your NON_WORKFLOW CSV lives)
LOCAL_INSTRU_DIR = BASE_RQ1 / "Local_Instru_Tests"
LOCAL_NON_WORKFLOW_CSV = LOCAL_INSTRU_DIR / "Repos_with_Local_GHA_Actions_NON_WORKFLOW.csv"

# TEMP git dirs per repo (no checkout)
CLONE_DIR = LOCAL_INSTRU_DIR / "Local_Cloned_Repos"

# Where we store extracted local YAML files (ACTION.YML/ACTION.YAML),
# preserving repo-relative paths:
#   D:\All_Action_YMLs\<owner.repo>\<relative_path_inside_repo>
OUTPUT_YML_DIR = BASE_RQ1 / "All_Action_YMLs"

# Index CSV describing extracted local YAMLs
LOCAL_INDEX_CSV = LOCAL_INSTRU_DIR / "Local_YML_Extraction_Index.csv"

# Whether to keep temp git dirs after extraction
KEEP_CLONES = False

# --- GitHub API behavior ---
REQUEST_TIMEOUT_SECONDS = 30

# If tokens are rate-limited, do you want to WAIT until reset?
WAIT_FOR_RATE_LIMIT_RESET = False

# Secondary limit backoff (short, bounded)
SECONDARY_BACKOFF_BASE_SECONDS = 10
SECONDARY_BACKOFF_MAX_SECONDS = 90
MAX_SECONDARY_RETRIES_PER_CALL = 3
MAX_TOTAL_API_ATTEMPTS_PER_CALL = 50

# Git on Windows
GIT_PREFIX = ["git", "-c", "core.longpaths=true"]


# ========= subprocess helpers =========

def run_text(cmd, cwd: Optional[Path] = None, check: bool = True) -> subprocess.CompletedProcess:
    """Run a command and capture stdout+stderr as text, safely decoded."""
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
        check=check,
    )


def run_bytes(cmd, cwd: Optional[Path] = None, check: bool = True) -> subprocess.CompletedProcess:
    """Run a command and capture stdout+stderr as bytes."""
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=False,
        check=check,
    )


def force_remove_readonly(func, path, _):
    os.chmod(path, stat.S_IWRITE)
    func(path)


def sanitize_token(s: str) -> str:
    return re.sub(r"[^a-z0-9._+-]", "_", s.lower())


# ========= branch detection =========

def detect_default_branch(repo_url: str) -> str:
    try:
        out = run_text(GIT_PREFIX + ["ls-remote", "--symref", repo_url, "HEAD"]).stdout
        for line in out.splitlines():
            s = line.strip()
            if s.startswith("ref: ") and s.endswith("HEAD"):
                ref = s.split()[1]  # refs/heads/main
                if ref.startswith("refs/heads/"):
                    return ref.split("/", 2)[2]
    except Exception:
        pass

    # fallback guesses
    for guess in ("main", "master"):
        try:
            run_text(GIT_PREFIX + ["ls-remote", repo_url, f"refs/heads/{guess}"], check=True)
            return guess
        except Exception:
            continue

    raise RuntimeError("Could not determine default branch via git ls-remote")


# ========= load tokens (notebook + script safe) =========

def find_env_file(env_dir: Path, env_filename: str) -> Path:
    """
    Try these locations (in order):
      1) ENV_DIR/env_filename (your stated location)
      2) cwd/env_filename
      3) BASE_RQ1/env_filename
      4) parents of cwd (up to 5 levels)
      5) script directory (if available)
    """
    candidates: list[Path] = []

    candidates.append(env_dir / env_filename)
    candidates.append(Path.cwd() / env_filename)
    candidates.append(BASE_RQ1 / env_filename)

    # walk up cwd a bit (helpful in notebooks)
    cur = Path.cwd()
    for _ in range(5):
        candidates.append(cur / env_filename)
        if cur.parent == cur:
            break
        cur = cur.parent

    # script dir (if running as a script)
    try:
        script_dir = Path(__file__).resolve().parent  # type: ignore[name-defined]
        candidates.append(script_dir / env_filename)
    except NameError:
        pass

    env_path = next((p for p in candidates if p.exists()), None)
    if env_path is None:
        raise FileNotFoundError(
            f"Could not find env file '{env_filename}'. Tried:\n" +
            "\n".join(f"  {p}" for p in candidates)
        )
    return env_path


def load_tokens(env_dir: Path, env_filename: str) -> list[str]:
    env_path = find_env_file(env_dir, env_filename)
    load_dotenv(dotenv_path=str(env_path), override=True)

    tokens: list[str] = []
    for i in range(1, 50):
        t = os.getenv(f"GITHUB_TOKEN_{i}")
        if t:
            tokens.append(t.strip())
    return tokens


TOKENS = load_tokens(ENV_DIR, ENV_FILE_NAME)


# ========= GitHub API: rotation + backoff =========

GITHUB_API_BASE_HEADERS = {
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}


@dataclass
class TokenState:
    remaining: Optional[int] = None
    reset_epoch: Optional[int] = None
    disabled: bool = False


@dataclass
class TokenPool:
    tokens: list[str]
    idx: int = 0
    state: Dict[str, TokenState] = field(default_factory=dict)

    def __post_init__(self):
        for t in self.tokens:
            self.state.setdefault(t, TokenState())

    def _is_usable(self, token: str) -> bool:
        st = self.state[token]
        if st.disabled:
            return False
        now = int(time.time())
        if st.remaining == 0 and st.reset_epoch and st.reset_epoch > now:
            return False
        return True

    def next_token(self) -> Optional[str]:
        if not self.tokens:
            return None
        for _ in range(len(self.tokens)):
            token = self.tokens[self.idx % len(self.tokens)]
            self.idx += 1
            if self._is_usable(token):
                return token
        return None

    def mark_rate_info(self, token: str, remaining: Optional[int], reset_epoch: Optional[int]) -> None:
        st = self.state[token]
        if remaining is not None:
            st.remaining = remaining
        if reset_epoch is not None:
            st.reset_epoch = reset_epoch

    def disable(self, token: str) -> None:
        self.state[token].disabled = True

    def earliest_reset(self) -> Optional[int]:
        resets = [
            st.reset_epoch
            for st in self.state.values()
            if st.reset_epoch is not None and st.remaining == 0 and not st.disabled
        ]
        return min(resets) if resets else None


SESSION = requests.Session()
POOL = TokenPool(TOKENS)


def _parse_int_header(resp: requests.Response, name: str) -> Optional[int]:
    v = resp.headers.get(name)
    if v is None:
        return None
    try:
        return int(v)
    except Exception:
        return None


def github_get(url: str, params: dict) -> requests.Response:
    secondary_tries = 0
    attempts = 0

    while attempts < MAX_TOTAL_API_ATTEMPTS_PER_CALL:
        attempts += 1

        token = POOL.next_token()
        headers = dict(GITHUB_API_BASE_HEADERS)
        if token:
            headers["Authorization"] = f"Bearer {token}"

        resp = SESSION.get(url, headers=headers, params=params, timeout=REQUEST_TIMEOUT_SECONDS)

        # Parse message if possible
        msg = ""
        try:
            j = resp.json()
            if isinstance(j, dict):
                msg = str(j.get("message", "") or "")
        except Exception:
            pass

        remaining = _parse_int_header(resp, "X-RateLimit-Remaining")
        reset = _parse_int_header(resp, "X-RateLimit-Reset")
        retry_after = _parse_int_header(resp, "Retry-After")

        if token:
            POOL.mark_rate_info(token, remaining, reset)

        if resp.status_code == 200:
            return resp

        if resp.status_code == 401:
            if token:
                print("  GitHub API: 401 Unauthorized; disabling that token.")
                POOL.disable(token)
            return resp

        if resp.status_code in (403, 429):
            low_msg = msg.lower()

            # secondary rate limit
            if "secondary rate limit" in low_msg:
                if secondary_tries >= MAX_SECONDARY_RETRIES_PER_CALL:
                    return resp
                wait_s = retry_after if retry_after is not None else (
                    SECONDARY_BACKOFF_BASE_SECONDS * (2 ** secondary_tries)
                )
                wait_s = min(wait_s, SECONDARY_BACKOFF_MAX_SECONDS)
                secondary_tries += 1
                print(f"  GitHub API: secondary rate limit; sleeping {wait_s}s then retrying...")
                time.sleep(wait_s)
                continue

            # primary rate limit exhausted
            if remaining == 0:
                # try other token
                next_tok = POOL.next_token()
                if next_tok is not None:
                    continue

                # all tokens exhausted
                if not WAIT_FOR_RATE_LIMIT_RESET:
                    return resp

                earliest = POOL.earliest_reset()
                if earliest is None:
                    return resp
                now = int(time.time())
                sleep_s = max(1, earliest - now)
                print(f"  GitHub API: all tokens exhausted; sleeping {sleep_s}s until reset...")
                time.sleep(sleep_s)
                continue

            return resp

        return resp

    return resp


def get_cutoff_commit(owner: str, repo: str, branch: str, cutoff_iso: str) -> Optional[str]:
    url = f"https://api.github.com/repos/{owner}/{repo}/commits"
    params = {"sha": branch, "until": cutoff_iso, "per_page": 1}

    resp = github_get(url, params=params)
    if resp.status_code != 200:
        try:
            j = resp.json()
            msg = j.get("message", "") if isinstance(j, dict) else ""
        except Exception:
            msg = (resp.text or "")[:200]

        print(
            f"  GitHub API error for {owner}/{repo} commits: {resp.status_code} {msg} "
            f"(remaining={resp.headers.get('X-RateLimit-Remaining')}, "
            f"reset={resp.headers.get('X-RateLimit-Reset')}, "
            f"retry_after={resp.headers.get('Retry-After')})"
        )
        return None

    data = resp.json()
    if not isinstance(data, list) or not data:
        return None
    return data[0].get("sha")


# ========= git read without checkout =========

def normalize_uses_path(uses_path: str) -> str:
    norm = uses_path.strip().strip('"').strip("'").replace("\\", "/")
    if norm.startswith("./"):
        norm = norm[2:]
    norm = norm.lstrip("/")
    norm = norm.rstrip("/")
    if norm in ("", "."):
        return ""  # repo root
    return norm


def git_path_exists(repo_dir: Path, commit_sha: str, rel_path: str) -> bool:
    if not rel_path:
        return False
    target = f"{commit_sha}:{rel_path}"
    proc = run_text(GIT_PREFIX + ["cat-file", "-e", target], cwd=repo_dir, check=False)
    return proc.returncode == 0


def git_show_file(repo_dir: Path, commit_sha: str, rel_path: str) -> Optional[bytes]:
    if not rel_path:
        return None
    target = f"{commit_sha}:{rel_path}"
    proc = run_bytes(GIT_PREFIX + ["show", target], cwd=repo_dir, check=False)
    if proc.returncode != 0:
        return None
    return proc.stdout


def resolve_action_yaml_path_at_commit(
    repo_dir: Path,
    commit_sha: str,
    uses_path: str
) -> Optional[str]:
    """
    Given a 'uses:' path (local to repo), return the repo-relative path
    of the actual action YAML at a specific commit, or None if not found.
    """
    norm = normalize_uses_path(uses_path)

    # direct YAML file
    if norm.lower().endswith((".yml", ".yaml")):
        return norm if git_path_exists(repo_dir, commit_sha, norm) else None

    # directory action.yml/action.yaml or root action.yml/action.yaml
    for fname in ("action.yml", "action.yaml"):
        candidate = f"{norm}/{fname}" if norm else fname
        if git_path_exists(repo_dir, commit_sha, candidate):
            return candidate

    return None


def prepare_git_dir(repo_git_dir: Path, github_url: str) -> None:
    if repo_git_dir.exists():
        shutil.rmtree(repo_git_dir, onerror=force_remove_readonly)
    repo_git_dir.mkdir(parents=True, exist_ok=True)
    run_text(GIT_PREFIX + ["init"], cwd=repo_git_dir)
    run_text(GIT_PREFIX + ["remote", "add", "origin", github_url], cwd=repo_git_dir)


def fetch_ref(repo_git_dir: Path, ref: str) -> str:
    run_text(GIT_PREFIX + ["fetch", "--depth", "1", "origin", ref], cwd=repo_git_dir)
    sha = run_text(GIT_PREFIX + ["rev-parse", "FETCH_HEAD"], cwd=repo_git_dir).stdout.strip()
    return sha


def save_action_file(owner_repo_key: str, relative_path: str, content: bytes) -> Path:
    """
    Save the action YAML preserving its repo-relative path:

        D:\All_Action_YMLs\<owner_repo_key>\<relative_path_inside_repo>

    Example:
        owner_repo_key = 'facebook.litho'
        relative_path  = '.github/actions/android-tests/action.yml'

    => 'D:\\All_Action_YMLs\\facebook.litho\\.github\\actions\\android-tests\\action.yml'
    """
    rel_norm = relative_path.replace("\\", "/").lstrip("/")
    dest = OUTPUT_YML_DIR / owner_repo_key / rel_norm
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_bytes(content)
    return dest


# ========= MAIN =========

def main():
    LOCAL_INSTRU_DIR.mkdir(parents=True, exist_ok=True)
    CLONE_DIR.mkdir(parents=True, exist_ok=True)
    OUTPUT_YML_DIR.mkdir(parents=True, exist_ok=True)

    print(f"Loaded {len(TOKENS)} GitHub token(s) from {ENV_DIR}\\{ENV_FILE_NAME}")
    if len(TOKENS) == 0:
        print("WARNING: Running unauthenticated -> you WILL hit API rate limits quickly.")

    # 1) Load URL list
    if not URL_LIST_CSV.exists():
        raise FileNotFoundError(f"URL list CSV not found: {URL_LIST_CSV}")

    url_df = pd.read_csv(URL_LIST_CSV)
    url_df.columns = url_df.columns.str.strip().str.lower()
    if "github_url" not in url_df.columns:
        raise ValueError("URL_List.csv must contain a 'github_url' column")

    owner_repo_to_info: Dict[str, Dict[str, str]] = {}
    for raw_url in url_df["github_url"].dropna().astype(str):
        raw_url = raw_url.strip()
        if not raw_url.startswith("http"):
            continue
        parsed = urlparse(raw_url)
        parts = parsed.path.strip("/").split("/")
        if len(parts) < 2:
            continue
        owner_raw, project_raw = parts[0], parts[1].replace(".git", "")
        key = f"{sanitize_token(owner_raw)}.{sanitize_token(project_raw)}"
        owner_repo_to_info[key] = {
            "github_url": raw_url,
            "owner": owner_raw,
            "project": project_raw,
        }

    print(f"Built mapping for {len(owner_repo_to_info)} repos from URL_List.csv")

    # 2) Load NON_WORKFLOW local uses CSV
    if not LOCAL_NON_WORKFLOW_CSV.exists():
        raise FileNotFoundError(f"Local NON_WORKFLOW CSV not found: {LOCAL_NON_WORKFLOW_CSV}")

    local_df = pd.read_csv(LOCAL_NON_WORKFLOW_CSV)
    local_df.columns = local_df.columns.str.strip()

    if "owner_repo" not in local_df.columns or "uses_paths" not in local_df.columns:
        raise ValueError(
            "Repos_with_Local_GHA_Actions_NON_WORKFLOW.csv must contain 'owner_repo' and 'uses_paths' columns"
        )

    local_uses_by_repo: Dict[str, Set[str]] = {}
    for _, row in local_df.iterrows():
        owner_repo = str(row["owner_repo"]).strip()
        uses_field = str(row["uses_paths"]).strip()
        if not owner_repo or not uses_field or uses_field.lower() == "nan":
            continue
        for raw_part in uses_field.split(";"):
            up = raw_part.strip()
            if up:
                local_uses_by_repo.setdefault(owner_repo, set()).add(up)

    print(f"Found {len(local_uses_by_repo)} repos with NON-workflow local uses")

    # 3) Prepare index CSV
    index_fields = [
        "owner",
        "repo",
        "owner_repo_key",
        "github_url",
        "default_branch",
        "snapshot_commit",   # actual fetched SHA (cutoff or HEAD)
        "local_use_path",
        "relative_path",
        "filename",
        "flat_filename",     # here: basename of the saved file
        "saved_to",
    ]
    if not LOCAL_INDEX_CSV.exists():
        with LOCAL_INDEX_CSV.open("w", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=index_fields).writeheader()

    # 4) Process repos
    for owner_repo_key, uses_set in sorted(local_uses_by_repo.items()):
        info = owner_repo_to_info.get(owner_repo_key)
        if not info:
            print(f"No URL found in URL_List.csv for key: {owner_repo_key} – skipping.")
            continue

        github_url = info["github_url"]
        owner = info["owner"]
        project = info["project"]

        print(f"\nProcessing repo {owner_repo_key} -> {github_url}")

        # default branch
        try:
            default_branch = detect_default_branch(github_url)
            print(f"  Default branch: {default_branch}")
        except Exception as e:
            print(f"  Failed to detect default branch: {e}")
            continue

        # cutoff commit via API; if none or API fails, fall back to HEAD branch
        cutoff_sha = get_cutoff_commit(owner, project, default_branch, CUTOFF_ISO)
        fetch_ref_name = cutoff_sha or default_branch

        if cutoff_sha:
            print(f"  Cutoff commit (API): {cutoff_sha}")
        else:
            print(f"  No cutoff commit (none before cutoff OR API failure). "
                  f"Falling back to HEAD of {default_branch}")

        # temp git dir
        repo_git_dir = CLONE_DIR / owner_repo_key.replace("/", "_")
        try:
            prepare_git_dir(repo_git_dir, github_url)
            snapshot_sha = fetch_ref(repo_git_dir, fetch_ref_name)
            print(f"  Fetched snapshot commit: {snapshot_sha}")
        except Exception as e:
            if isinstance(e, subprocess.CalledProcessError):
                print(f"  Failed to prepare/fetch repo:\n{e.stdout}")
            else:
                print(f"  Failed to prepare/fetch repo: {e}")
            continue

        # resolve & save each local use path
        for local_path in sorted(uses_set):
            print(f"    Resolving local use path: {local_path}")

            rel_file_path = resolve_action_yaml_path_at_commit(
                repo_git_dir, snapshot_sha, local_path
            )
            if not rel_file_path:
                print(f"    Could not resolve YAML for local path in snapshot: {local_path}")
                continue

            content = git_show_file(repo_git_dir, snapshot_sha, rel_file_path)
            if content is None:
                print(f"    Could not read file content via git show: {rel_file_path}")
                continue

            # Save preserving repo-relative folder structure
            dest_path = save_action_file(owner_repo_key, rel_file_path, content)

            filename = Path(rel_file_path).name

            with LOCAL_INDEX_CSV.open("a", newline="", encoding="utf-8") as f:
                writer = csv.DictWriter(f, fieldnames=index_fields)
                writer.writerow({
                    "owner": owner,
                    "repo": project,
                    "owner_repo_key": owner_repo_key,
                    "github_url": github_url,
                    "default_branch": default_branch,
                    "snapshot_commit": snapshot_sha,
                    "local_use_path": local_path,
                    "relative_path": rel_file_path,
                    "filename": filename,
                    "flat_filename": dest_path.name,  # basename of the saved file
                    "saved_to": str(dest_path),
                })

            print(f"    Saved {rel_file_path} as {dest_path}")

        if not KEEP_CLONES:
            try:
                shutil.rmtree(repo_git_dir, onerror=force_remove_readonly)
            except Exception as e:
                print(f"  Failed to delete temp git dir {repo_git_dir}: {e}")

    print("\nDone. Local YAMLs stored in:")
    print(f"  {OUTPUT_YML_DIR}")
    print("Index CSV:")
    print(f"  {LOCAL_INDEX_CSV}")


if __name__ == "__main__":
    main()


Loaded 4 GitHub token(s) from C:\GitHub\Android-Mobile-Apps\All_Tokens.env
Built mapping for 4697 repos from URL_List.csv
Found 160 repos with NON-workflow local uses

Processing repo 1q23lyc45.kitsunemagisk -> https://github.com/1q23lyc45/KitsuneMagisk
  Default branch: kitsune
  Cutoff commit (API): b030f742cbb02c01c54f250555fea16b5389fea8
  Fetched snapshot commit: b030f742cbb02c01c54f250555fea16b5389fea8
    Resolving local use path: ./.github/actions/setup
    Saved .github/actions/setup/action.yml as D:\temp\All_Action_YMLs\1q23lyc45.kitsunemagisk\.github\actions\setup\action.yml

Processing repo 4accccc.vivo-magisk-suu -> https://github.com/4accccc/vivo-Magisk-suu
  Default branch: master
  Cutoff commit (API): e53b0b8e2c02e3f492310820d2156c3efa3664ec
  Fetched snapshot commit: e53b0b8e2c02e3f492310820d2156c3efa3664ec
    Resolving local use path: ./.github/actions/setup
    Saved .github/actions/setup/action.yml as D:\temp\All_Action_YMLs\4accccc.vivo-magisk-suu\.github\actions